# Hitting the Vibe: re-implementing the algorithm (November 2026)

**What this is:** a re-implementation of the algorithm from a Spring 2024 group project in NYU's *Data Analysis in the Music Industry* course (Fred Xiang, Owen Tynes, Sam Richter, Gerrit Wendt), following the steps described in the final paper. The goal is to understand each step and to test the paper's two conclusions with data.

**Differences from the original project:**
- Spotify closed its audio-features API to new apps in late 2024, so this uses the public TidyTuesday `spotify_songs` dataset (about 28,000 songs with Spotify audio features, playlist names and playlist genres) instead of the live API.
- The original song pool combined the Billboard Hot 200 and the Billboard Global all-time 100 (300 songs); here it is the 300 most popular songs in the dataset.

**Structure:** steps 1–4 re-implement the original algorithm; step 5 tests the paper's two conclusions; step 6 is an issue found during the re-implementation.

In [1]:
import pandas as pd, numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

raw = pd.read_csv("spotify_songs.csv")
# A song can appear in several playlists: keep one row per song,
# but remember whether ANY of its playlists has "party" in the name (used in step 5).
in_party = raw.groupby("track_id").playlist_name.apply(lambda s: s.str.contains("party", case=False).any())
songs = raw.drop_duplicates("track_id").set_index("track_id")
songs["in_party_playlist"] = in_party
print(f"{len(raw):,} rows -> {len(songs):,} unique songs")

32,833 rows -> 28,356 unique songs


## Step 1: build playlists for 10 simulated guests

Original paper: put 300 popular songs together and use random numbers to draw 25 songs for each of 10 guests, standing in for their Spotify top 25.

In [2]:
# Many songs tie on popularity at the cutoff (57 songs share the 300th value), so break ties by
# track_id with a stable sort; otherwise the pool, and every result after it, can differ between machines.
pool = (songs.reset_index()
             .sort_values(["track_popularity", "track_id"], ascending=[False, True], kind="mergesort")
             .set_index("track_id").head(300))
rng = np.random.default_rng(42)
guests = {f"Guest {i+1}": pool.sample(25, random_state=int(rng.integers(1_000_000))) for i in range(10)}
print({k: len(v) for k, v in guests.items()})

{'Guest 1': 25, 'Guest 2': 25, 'Guest 3': 25, 'Guest 4': 25, 'Guest 5': 25, 'Guest 6': 25, 'Guest 7': 25, 'Guest 8': 25, 'Guest 9': 25, 'Guest 10': 25}


## Step 2: give every song a party score and keep each guest's top 10

The paper's formula: **party score = danceability + energy − acousticness**. Instead of a fixed threshold, each guest contributes their 10 highest-scoring songs.

In [3]:
songs["party_score"] = songs.danceability + songs.energy - songs.acousticness
for g in guests.values():
    g["party_score"] = g.danceability + g.energy - g.acousticness
candidates = pd.concat([g.nlargest(10, "party_score") for g in guests.values()])
candidates = candidates[~candidates.index.duplicated()]   # same hit can be in several guests' lists
print(len(candidates), "candidate songs after removing duplicates")

77 candidate songs after removing duplicates


## Step 3: use cosine similarity to pick the 25 songs that fit together best

Original paper: build a matrix from six features already scaled 0–1 (danceability, energy, speechiness, instrumentalness, acousticness, valence) and compute the cosine similarity of every pair of songs. The higher a song's average similarity to all the others, the better it fits the group. The 25 songs with the highest average similarity form the playlist.

In [4]:
FEATURES = ["danceability", "energy", "speechiness", "instrumentalness", "acousticness", "valence"]

def build_mix(cands, X, n=25):
    sim = cosine_similarity(X)
    np.fill_diagonal(sim, np.nan)          # ignore each song's similarity with itself
    out = cands.copy()
    out["mean_sim"] = np.nanmean(sim, axis=1)
    return out.nlargest(n, "mean_sim")

mix = build_mix(candidates, candidates[FEATURES])
mix[["track_name", "track_artist", "playlist_genre", "party_score", "mean_sim"]].round(3)

,track_name,track_artist,playlist_genre,party_score,mean_sim
track_id,,,,,
7rknFhrVDyxzTJkPSf7LoW,No Me Conoce - Remix,Jhay Cortez,rap,1.440,0.976
7ef4DlsgrMEH11cDZd32M6,One Kiss (with Dua Lipa),Calvin Harris,pop,1.616,0.975
58q2HKrzhC3ozto2nDdN4z,I Like It,Cardi B,pop,1.443,0.975
0pqnGHJpmpxLKifKRmU6WP,Believer,Imagine Dragons,latin,1.494,0.975
2BgEsaKNfHUdlh97KmvFyo,2002,Anne-Marie,pop,1.343,0.974
059bcIhyc2SBwm6sw2AZzd,Te Vi,Piso 21,latin,1.609,0.974
6WrI0LAC5M1Rw2MnX2ZvEg,Don't Start Now,Dua Lipa,pop,1.575,0.973
4w8niZpiMy6qz1mntFA5uM,"Taki Taki (with Selena Gomez, Ozuna & Cardi B)",DJ Snake,pop,1.486,0.973
25sgk305KZfyuqVBQIahim,Sweet but Psycho,Ava Max,pop,1.354,0.973


## Step 4: overview of the result

In [5]:
print("Genres in the mix:", mix.playlist_genre.value_counts().to_dict())
print("Mean-similarity range of all candidates:",
      round(build_mix(candidates, candidates[FEATURES], n=len(candidates)).mean_sim.min(), 3), "to",
      round(mix.mean_sim.max(), 3))

Genres in the mix: {'pop': 15, 'rap': 5, 'latin': 5}
Mean-similarity range of all candidates: 0.885 to 0.976


## Step 5: testing the paper's two conclusions

**Conclusion A: audio features alone cannot set a threshold that picks out "party" songs.**

Test: use "appears in a playlist whose name contains *party*" as a rough party-song label, and measure how well each score separates party songs from the rest. The metric is **AUC**: 0.5 is guessing, 1.0 is perfect separation.

In [6]:
y = songs.in_party_playlist
print(f"Party-playlist songs: {y.sum()} of {len(y):,} ({y.mean():.1%})")
for col in ["danceability", "energy", "party_score"]:
    print(f"AUC of {col:12s}: {roc_auc_score(y, songs[col]):.3f}")

Party-playlist songs: 670 of 28,356 (2.4%)
AUC of danceability: 0.481
AUC of energy      : 0.583
AUC of party_score : 0.596


**Conclusion B: audio features do not predict genre well.**

Test: train a random forest on the six features to predict six genres (pop, rap, rock, latin, r&b, edm) with 5-fold cross-validation, and compare it with always guessing the most common genre.

In [7]:
sample = songs.sample(8000, random_state=0)
rf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=0)
acc = cross_val_score(rf, sample[FEATURES], sample.playlist_genre, cv=5).mean()
baseline = sample.playlist_genre.value_counts(normalize=True).max()
print(f"Genre accuracy from audio features: {acc:.1%}  (majority-class baseline: {baseline:.1%})")

Genre accuracy from audio features: 47.2%  (majority-class baseline: 19.0%)


## Step 6: an issue found during the re-implementation — the result is sensitive to preprocessing

In step 4, every candidate's average similarity is above 0.88. All six features are positive, so any two songs' feature vectors point in similar directions and cosine similarity is high by construction. The ranking then depends on very small differences.

A common alternative is to **standardize** each feature first (subtract the mean, divide by the standard deviation) and then compute cosine similarity, which compares how each song sits above or below average. Below: how much the playlist changes under this other preprocessing.

In [8]:
X_std = StandardScaler().fit_transform(candidates[FEATURES])
raw_all = build_mix(candidates, candidates[FEATURES], n=len(candidates)).mean_sim
std_all = build_mix(candidates, X_std, n=len(candidates)).mean_sim
print(f"Raw features:          mean_sim spread = {raw_all.max()-raw_all.min():.3f}")
print(f"Standardized features: mean_sim spread = {std_all.max()-std_all.min():.3f}")
mix_std = build_mix(candidates, X_std)
print("Songs that change between the two mixes:", len(set(mix.index) ^ set(mix_std.index)) // 2, "of 25")

Raw features:          mean_sim spread = 0.090
Standardized features: mean_sim spread = 0.149
Songs that change between the two mixes: 10 of 25


## Summary

**What the data say about the paper's conclusions:**
- **Conclusion A holds, even more clearly than stated.** Danceability separates party songs with an AUC below 0.5, no better than chance; the party-score formula does somewhat better but is still weak. The paper's decision to rank instead of using a threshold was right.
- **Conclusion B needs more precise wording.** The features predict genre clearly better than guessing, but far from reliably. More accurately: audio features **carry some** genre information, but not enough to separate genres.

**A method issue found in the re-implementation:** on raw features, cosine similarity is high for everyone (all above 0.88), so nearly every song "fits". Switching to standardized features alone replaces 10 of the 25 songs, so the final playlist depends heavily on a preprocessing choice the paper did not discuss.

**Limitations:** the party label comes from playlist names and is rough; the genre is the playlist's genre, not necessarily the song's (for example, *Believer* is labeled latin).

**Link to my research direction:** surface audio features struggle to capture deeper properties of music (whether it suits a party, which genre it belongs to). This is the same question as finding the right features for detecting AI music.